# 03_03: Audio Classification

**Objective:** Learn to classify audio signals into categories — environmental sounds, music genres, speech commands, and more.

**Prerequisites:** Basic Python, familiarity with HuggingFace pipelines (Notebook 00_03). No audio processing experience needed.

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|-------------|------------|------|---------|-------------------|-------|
| **Small (CPU)** | MIT/ast-finetuned-audioset-10-10-0.4593 | ~350MB | 4GB RAM | Any CPU | 527 AudioSet classes |
| **Alternative** | superb/wav2vec2-base-superb-ks | ~360MB | 4GB RAM | Any CPU | Keyword spotting |

## Expected Behaviors

### First Time Running
- **Model download**: ~350MB for AST model (1-3 minutes)
- Audio sample downloads are small (~100KB each)

### What You'll See
- Models classify short audio clips into hundreds of categories
- Environmental sounds (dog barking, car horn, music) are well-recognized
- The Audio Spectrogram Transformer (AST) converts audio to spectrograms then uses a Vision Transformer

### Common Observations
- Short, clean clips get high confidence scores
- Noisy or overlapping sounds may yield multiple plausible labels
- Sample rate (16kHz vs 44.1kHz) affects results

## Overview

**Audio classification** assigns a label to an audio signal. Unlike speech recognition (which transcribes words), audio classification identifies *what kind of sound* is present.

### Common Audio Classification Tasks

| Task | Description | Example Labels |
|------|-------------|---------------|
| **Environmental Sound** | Classify ambient sounds | Dog bark, siren, rain, thunder |
| **Music Genre** | Identify music style | Rock, jazz, classical, hip-hop |
| **Keyword Spotting** | Detect specific words | "yes", "no", "stop", "go" |
| **Speaker Verification** | Verify speaker identity | Same/different speaker |
| **Emotion Recognition** | Detect emotion from voice | Happy, sad, angry, neutral |

### How Audio Models Work

Most modern audio classifiers follow this pipeline:
1. **Waveform** → raw audio signal (1D array of amplitudes)
2. **Feature extraction** → convert to spectrogram or mel-spectrogram (2D image-like representation)
3. **Model** → Transformer or CNN processes the features
4. **Classification** → output probabilities over classes

## Setup and Installation

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import pipeline, AutoFeatureExtractor, AutoModelForAudioClassification
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Check optional audio libraries
try:
    import librosa
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False
    print('librosa not installed. Install with: pip install librosa')

try:
    import soundfile as sf
    SOUNDFILE_AVAILABLE = True
except ImportError:
    SOUNDFILE_AVAILABLE = False
    print('soundfile not installed. Install with: pip install soundfile')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if LIBROSA_AVAILABLE:
    print(f'librosa: {librosa.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Selection

In [ ]:
# Option 1: Audio Spectrogram Transformer (recommended)
MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'  # ~350MB, 527 classes

# Option 2: Wav2Vec2 for keyword spotting
# MODEL_NAME = 'superb/wav2vec2-base-superb-ks'  # ~360MB, 12 keywords

# Target sample rate for the model
TARGET_SAMPLE_RATE = 16000

## Generating Audio Samples

Since we want this notebook to run without external audio files, we'll generate synthetic audio signals for demonstration. These simple waveforms (tones, noise, patterns) will let us test the classifier even without real audio recordings.

In [ ]:
def generate_sine_wave(
    frequency: float = 440.0,
    duration: float = 3.0,
    sample_rate: int = 16000,
    amplitude: float = 0.5,
) -> np.ndarray:
    """Generate a sine wave audio signal.

    Args:
        frequency: Frequency in Hz.
        duration: Duration in seconds.
        sample_rate: Samples per second.
        amplitude: Signal amplitude (0.0 to 1.0).

    Returns:
        NumPy array of audio samples.
    """
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    return (amplitude * np.sin(2 * np.pi * frequency * t)).astype(np.float32)


def generate_white_noise(
    duration: float = 3.0,
    sample_rate: int = 16000,
    amplitude: float = 0.3,
) -> np.ndarray:
    """Generate white noise audio signal.

    Args:
        duration: Duration in seconds.
        sample_rate: Samples per second.
        amplitude: Signal amplitude.

    Returns:
        NumPy array of audio samples.
    """
    return (amplitude * np.random.randn(int(sample_rate * duration))).astype(np.float32)


def generate_chirp(
    start_freq: float = 200.0,
    end_freq: float = 2000.0,
    duration: float = 3.0,
    sample_rate: int = 16000,
) -> np.ndarray:
    """Generate a frequency sweep (chirp) signal.

    Args:
        start_freq: Starting frequency in Hz.
        end_freq: Ending frequency in Hz.
        duration: Duration in seconds.
        sample_rate: Samples per second.

    Returns:
        NumPy array of audio samples.
    """
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    freq = np.linspace(start_freq, end_freq, len(t))
    phase = 2 * np.pi * np.cumsum(freq) / sample_rate
    return (0.5 * np.sin(phase)).astype(np.float32)


# Generate test signals
audio_samples = {
    'Sine 440Hz (A4 note)': generate_sine_wave(440.0),
    'Sine 880Hz (A5 note)': generate_sine_wave(880.0),
    'White Noise': generate_white_noise(),
    'Chirp (200-2000Hz)': generate_chirp(),
}

print(f'Generated {len(audio_samples)} audio samples:')
for name, audio in audio_samples.items():
    print(f'  {name:25s}: {len(audio):,} samples ({len(audio)/TARGET_SAMPLE_RATE:.1f}s)')

In [ ]:
# Visualize the audio waveforms
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, (name, audio) in enumerate(audio_samples.items()):
    # Show first 0.05s for detail
    display_samples = int(0.05 * TARGET_SAMPLE_RATE)
    t = np.arange(display_samples) / TARGET_SAMPLE_RATE * 1000  # ms
    axes[idx].plot(t, audio[:display_samples], linewidth=0.5)
    axes[idx].set_title(name, fontsize=11)
    axes[idx].set_xlabel('Time (ms)')
    axes[idx].set_ylabel('Amplitude')
    axes[idx].set_ylim(-1, 1)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Audio Waveforms (first 50ms)', fontsize=13)
plt.tight_layout()
plt.show()

## Method 1: Pipeline API

The audio classification pipeline handles feature extraction (converting waveforms to spectrograms) and model inference. You provide a raw audio array and sample rate, and it returns class predictions.

In [ ]:
# Create audio classification pipeline
audio_classifier = pipeline(
    'audio-classification',
    model=MODEL_NAME,
    device=device,
)

# Classify each synthetic audio sample
print('=== Audio Classification Results ===')
all_results: list[dict[str, str]] = []

for name, audio in audio_samples.items():
    predictions = audio_classifier(
        {'raw': audio, 'sampling_rate': TARGET_SAMPLE_RATE},
        top_k=3,
    )
    top_pred = predictions[0]
    all_results.append({
        'Audio': name,
        'Top-1 Label': top_pred['label'],
        'Score': f'{top_pred["score"]:.4f}',
        'Top-2': predictions[1]['label'] if len(predictions) > 1 else 'N/A',
        'Top-3': predictions[2]['label'] if len(predictions) > 2 else 'N/A',
    })

pd.DataFrame(all_results)

Synthetic tones and noise are useful for testing, but real-world audio classification shines with actual recordings. The AudioSet-trained AST model recognizes 527 categories covering music, speech, environmental sounds, and more. Let's try loading a real audio dataset for more realistic testing.

In [ ]:
# Try loading a real audio sample from the datasets library
try:
    from datasets import load_dataset, Audio
    
    # Load a small speech commands dataset
    speech_dataset = load_dataset(
        'google/speech_commands', 'v0.01',
        split='validation[:5]',
        trust_remote_code=True,
    )
    
    print(f'Loaded {len(speech_dataset)} speech command samples')
    print(f'Features: {list(speech_dataset.features.keys())}')
    print(f'Labels: {set(speech_dataset["label"][:5])}')
    
    REAL_AUDIO_AVAILABLE = True
except Exception as error:
    print(f'Could not load speech dataset: {error}')
    print('Continuing with synthetic audio only.')
    REAL_AUDIO_AVAILABLE = False

## Method 2: Manual Model Loading

Loading the model manually gives us access to the raw logits and lets us inspect how the feature extractor converts waveforms into the format the model expects.

In [ ]:
# Load model and feature extractor
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

id2label = model.config.id2label
print(f'Model: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Number of classes: {len(id2label)}')
print(f'Expected sample rate: {feature_extractor.sampling_rate}')
print(f'\nSample classes: {[id2label[i] for i in range(5)]}')

In [ ]:
def classify_audio_manual(
    audio: np.ndarray,
    sample_rate: int,
    feature_extractor: transformers.FeatureExtractionMixin,
    model: torch.nn.Module,
    id2label: dict[int, str],
    top_k: int = 5,
) -> pd.DataFrame:
    """Classify audio using manual model inference.

    Args:
        audio: Raw audio waveform as NumPy array.
        sample_rate: Sample rate of the audio.
        feature_extractor: Model's feature extractor.
        model: Audio classification model.
        id2label: Class ID to label mapping.
        top_k: Number of top predictions to return.

    Returns:
        DataFrame with top-k predictions.
    """
    inputs = feature_extractor(
        audio, sampling_rate=sample_rate, return_tensors='pt'
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    top_probs, top_indices = torch.topk(probs, top_k)
    
    rows: list[dict[str, str]] = []
    for prob, idx in zip(top_probs, top_indices):
        rows.append({
            'Rank': str(len(rows) + 1),
            'Label': id2label[idx.item()],
            'Score': f'{prob.item():.4f}',
        })
    
    return pd.DataFrame(rows)


# Classify the chirp signal with top-5 predictions
print('=== Detailed Classification: Chirp Signal ===')
classify_audio_manual(
    audio_samples['Chirp (200-2000Hz)'],
    TARGET_SAMPLE_RATE,
    feature_extractor, model, id2label,
    top_k=5,
)

## Understanding Spectrograms

The Audio Spectrogram Transformer (AST) works by converting audio waveforms into mel-spectrograms — 2D images where the x-axis is time, the y-axis is frequency, and the color intensity represents energy. This transforms the audio classification problem into an image classification problem, which is why the model uses a Vision Transformer (ViT) backbone.

In [ ]:
def plot_spectrogram(
    audio: np.ndarray,
    sample_rate: int,
    title: str = 'Spectrogram',
) -> None:
    """Plot a mel-spectrogram of an audio signal.

    Args:
        audio: Raw audio waveform.
        sample_rate: Sample rate of the audio.
        title: Plot title.
    """
    if not LIBROSA_AVAILABLE:
        print('librosa required for spectrogram visualization. Skipping.')
        return
    
    mel_spec = librosa.feature.melspectrogram(
        y=audio, sr=sample_rate, n_mels=128, fmax=8000,
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(
        mel_spec_db, sr=sample_rate,
        x_axis='time', y_axis='mel',
        cmap='viridis',
    )
    plt.colorbar(format='%+2.0f dB')
    plt.title(title, fontsize=13)
    plt.tight_layout()
    plt.show()


# Visualize spectrograms for each sample
for name, audio in audio_samples.items():
    plot_spectrogram(audio, TARGET_SAMPLE_RATE, title=f'Mel-Spectrogram: {name}')

The spectrograms reveal the different characteristics of each signal:
- **Sine waves** appear as horizontal lines at their frequency
- **White noise** has energy spread uniformly across all frequencies
- **Chirps** show a diagonal line sweeping from low to high frequency

This visual representation is what the AST model "sees" when classifying audio. Understanding spectrograms helps you interpret why the model makes certain predictions.

## Practical Applications

Audio classification is used in smart home devices (detecting glass breaking, alarms), wildlife monitoring (bird species identification), industrial quality control (detecting machine faults by sound), and accessibility features (alerting deaf users to environmental sounds).

In [ ]:
# Application: Multi-signal comparison
def compare_audio_signals(
    audio_dict: dict[str, np.ndarray],
    sample_rate: int,
    classifier_pipe: transformers.Pipeline,
) -> pd.DataFrame:
    """Compare classification results across multiple audio signals.

    Args:
        audio_dict: Dict mapping signal names to audio arrays.
        sample_rate: Sample rate of all signals.
        classifier_pipe: Audio classification pipeline.

    Returns:
        DataFrame comparing top predictions for each signal.
    """
    rows: list[dict[str, str]] = []
    for name, audio in audio_dict.items():
        preds = classifier_pipe(
            {'raw': audio, 'sampling_rate': sample_rate}, top_k=3
        )
        rows.append({
            'Signal': name,
            'Duration (s)': f'{len(audio) / sample_rate:.1f}',
            '#1': f'{preds[0]["label"]} ({preds[0]["score"]:.2f})',
            '#2': f'{preds[1]["label"]} ({preds[1]["score"]:.2f})' if len(preds) > 1 else 'N/A',
            '#3': f'{preds[2]["label"]} ({preds[2]["score"]:.2f})' if len(preds) > 2 else 'N/A',
        })
    return pd.DataFrame(rows)


# Generate more varied signals for comparison
extended_samples = {
    **audio_samples,
    'Low tone (100Hz)': generate_sine_wave(100.0),
    'High tone (4000Hz)': generate_sine_wave(4000.0),
    'Tone + Noise': generate_sine_wave(440.0) + generate_white_noise(amplitude=0.1),
}

print('=== Multi-Signal Classification Comparison ===')
compare_audio_signals(extended_samples, TARGET_SAMPLE_RATE, audio_classifier)

## Performance Benchmarking

Let's measure classification speed for different audio durations.

In [ ]:
def benchmark_audio_classification(
    durations: list[float],
    sample_rate: int,
    classifier_pipe: transformers.Pipeline,
    num_runs: int = 5,
) -> pd.DataFrame:
    """Benchmark classification speed for different audio durations.

    Args:
        durations: List of audio durations in seconds.
        sample_rate: Sample rate.
        classifier_pipe: Audio classification pipeline.
        num_runs: Number of runs for averaging.

    Returns:
        DataFrame with timing results.
    """
    results: list[dict[str, str]] = []
    
    for duration in durations:
        audio = generate_sine_wave(440.0, duration=duration, sample_rate=sample_rate)
        audio_input = {'raw': audio, 'sampling_rate': sample_rate}
        
        # Warmup
        classifier_pipe(audio_input, top_k=1)
        
        times: list[float] = []
        for _ in range(num_runs):
            start = time.perf_counter()
            classifier_pipe(audio_input, top_k=1)
            times.append(time.perf_counter() - start)
        
        results.append({
            'Duration (s)': f'{duration:.1f}',
            'Samples': f'{len(audio):,}',
            'Mean Latency (ms)': f'{np.mean(times) * 1000:.1f}',
            'Std (ms)': f'{np.std(times) * 1000:.1f}',
            'Device': str(device),
        })
    
    return pd.DataFrame(results)


print('=== Classification Speed vs Audio Duration ===')
benchmark_audio_classification(
    [1.0, 3.0, 5.0, 10.0],
    TARGET_SAMPLE_RATE,
    audio_classifier,
)

## Exercises

1. **Real audio**: Record or download a short audio clip (speech, music, or environmental sound) and classify it. Compare the results with your expectation.

2. **Noise robustness**: Add increasing amounts of white noise to a sine wave and see how the classification changes. At what noise level does the model lose its prediction?

3. **Music vs speech**: Generate or find samples of music and speech. Does the model reliably distinguish between them?

4. **Keyword spotting**: Load `superb/wav2vec2-base-superb-ks` and test it on the speech commands dataset. Compare accuracy and speed with the AST model.

## Key Takeaways

- **Audio classification** assigns labels to audio signals — environmental sounds, music genres, speech commands, etc.
- The **Audio Spectrogram Transformer (AST)** converts waveforms to spectrograms and applies a Vision Transformer
- **Mel-spectrograms** are the key feature representation — they transform 1D waveforms into 2D images
- Audio models expect a specific **sample rate** (typically 16kHz) — resample your audio if needed
- **527 AudioSet classes** cover a huge range of real-world sounds for the AST model

## Next Steps & Resources

**Next section**: [04_01 — Image-to-Text](../04_multimodal/04_01_multimodal_image_to_text.ipynb) — move to multimodal AI combining vision and language.

**Resources:**
- [AST Paper](https://arxiv.org/abs/2104.01778) — Audio Spectrogram Transformer
- [AudioSet](https://research.google.com/audioset/) — Google's large-scale audio dataset
- [HuggingFace Audio Classification Guide](https://huggingface.co/docs/transformers/tasks/audio_classification)
- [Audio Classification Models on Hub](https://huggingface.co/models?pipeline_tag=audio-classification)
- [librosa Documentation](https://librosa.org/) — Python library for audio analysis